In [2]:
"""
Stage 1 — aggregate.py
Reads all JSON files in a folder and builds a raw keyword aggregate.
No embedding / clustering happens here.

Usage:
    python aggregate.py <input_folder> [raw_output_file]

Example:
    python aggregate.py ./papers raw_aggregate.json
"""

import os
import json
import re
import sys
from typing import Any, List


# ── Helpers ────────────────────────────────────────────────────────────────────

def canonical_kw(k: str) -> str:
    return re.sub(r"\s+", " ", (k or "").strip()).lower()

def canonical_country(c: str) -> str:
    s = (c or "").strip()
    return s.title() if s else s

def canonical_author(a: str) -> str:
    s = (a or "").strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)
    return re.sub(r"\s+", " ", s).strip()

def extract_year(item: dict):
    y = item.get("Year") or item.get("year")
    if y:
        try:
            return str(int(y))
        except Exception:
            pass
    text = (item.get("title", "") or "") + " " + (item.get("abstract", "") or "")
    m = re.search(r"\b(19|20)\d{2}\b", text)
    return m.group(0) if m else None

def authors_from_authors_field(s: Any) -> List[str]:
    parts: List[str] = []
    if s is None:
        return parts
    if isinstance(s, list):
        for elt in s:
            parts.extend(authors_from_authors_field(elt))
        return [p for p in (x.strip() for x in parts) if p]
    if isinstance(s, dict):
        if "authors_structured" in s and isinstance(s["authors_structured"], list):
            for a in s["authors_structured"]:
                if isinstance(a, dict):
                    name = (a.get("name") or "").strip()
                    aff  = (a.get("affiliation") or "").strip()
                    combined = f"{name} ({aff})" if aff else name
                    if combined.strip():
                        parts.append(combined.strip())
                else:
                    parts.extend(authors_from_authors_field(a))
            return [p for p in (x.strip() for x in parts) if p]
        name = (s.get("name") or s.get("author") or "").strip()
        aff  = (s.get("affiliation") or s.get("affil") or "").strip()
        if name or aff:
            combined = f"{name} ({aff})" if aff else name
            return [combined.strip()] if combined.strip() else []
        s = json.dumps(s)
    s = str(s).strip()
    if not s:
        return []
    if any(sep in s for sep in (";", "\n", "|")):
        return [p.strip() for p in re.split(r";|\n|\|", s) if p.strip()]
    return [s]


# ── Core ───────────────────────────────────────────────────────────────────────

def process_folder(input_folder: str, output_file: str):
    json_files = sorted(f for f in os.listdir(input_folder) if f.endswith(".json"))
    if not json_files:
        raise SystemExit(f"No JSON files found in: {input_folder}")

    print(f"Found {len(json_files)} file(s) in '{input_folder}'")

    raw_agg = {}
    # Structure per keyword:
    # {
    #   "total_count":    int,                         <- used by cluster.py to pick canonical
    #   "years":          { year_str: count },
    #   "countries":      { country:  count },
    #   "authors":        { author:   count },
    #   "papers_by_year": { year_str: [ {title, source_file, countries, authors} ] }
    # }

    for filename in json_files:
        filepath = os.path.join(input_folder, filename)
        print(f"  Processing: {filename}")

        with open(filepath, "r", encoding="utf-8") as f:
            try:
                items = json.load(f)
            except json.JSONDecodeError as e:
                print(f"    ⚠ Skipping (JSON error): {e}")
                continue

        if isinstance(items, dict):
            items = [items]

        for item in items:
            year = extract_year(item)

            kws = item.get("keywords") or []
            if isinstance(kws, str):
                kws = [kws]

            countries_raw = item.get("countries") or []
            if isinstance(countries_raw, str):
                countries_raw = [countries_raw]
            countries = [canonical_country(c) for c in countries_raw if c]

            authors_field = (
                item.get("authors") or item.get("author")
                or item.get("authors_field") or ""
            )
            authors_clean = [
                canonical_author(a)
                for a in authors_from_authors_field(authors_field)
            ]

            paper_ref = {
                "title":       (item.get("title") or "").strip(),
                "source_file": filename,
                "countries":   countries,
                "authors":     authors_clean,
            }

            for rawk in kws:
                if not rawk:
                    continue
                k = canonical_kw(rawk)
                if not k:
                    continue

                rec = raw_agg.setdefault(k, {
                    "total_count":    0,
                    "years":          {},
                    "countries":      {},
                    "authors":        {},
                    "papers_by_year": {},
                })

                rec["total_count"] += 1

                if year:
                    rec["years"][year] = rec["years"].get(year, 0) + 1
                    rec["papers_by_year"].setdefault(year, []).append(paper_ref)

                for c in countries:
                    rec["countries"][c] = rec["countries"].get(c, 0) + 1

                for a in authors_clean:
                    if a:
                        rec["authors"][a] = rec["authors"].get(a, 0) + 1

    # Sort before saving
    for rec in raw_agg.values():
        try:
            rec["years"] = dict(sorted(rec["years"].items(), key=lambda x: int(x[0])))
        except Exception:
            rec["years"] = dict(sorted(rec["years"].items()))
        try:
            rec["papers_by_year"] = dict(
                sorted(rec["papers_by_year"].items(), key=lambda x: int(x[0]))
            )
        except Exception:
            rec["papers_by_year"] = dict(sorted(rec["papers_by_year"].items()))
        rec["countries"] = dict(sorted(rec["countries"].items(), key=lambda x: -x[1]))
        rec["authors"]   = dict(sorted(rec["authors"].items(),   key=lambda x: -x[1]))

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(raw_agg, f, ensure_ascii=False, indent=2)

    print(f"\n✓ Stage 1 complete!")
    print(f"  Unique keywords : {len(raw_agg)}")
    print(f"  Raw output      : {output_file}")
    print(f"\nNext step → run: python cluster.py {output_file}")


# ── Entry point ────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python aggregate.py <input_folder> [raw_output_file]")
        print("Example: python aggregate.py ./papers raw_aggregate.json")
        sys.exit(1)


    process_folder("groq/extracted_results", "aggregate.json")

Found 61 file(s) in 'groq/extracted_results'
  Processing: aiche_papers_3288_extracted_groq.json
  Processing: ext_aiche_papers_3298_p1.json
  Processing: ext_aiche_papers_3298_p2.json
  Processing: ext_aiche_papers_3298_p3.json
  Processing: ext_aiche_papers_3298_p4.json
  Processing: ext_aiche_papers_3298_p5.json
  Processing: ext_aiche_papers_3298_p6.json
  Processing: ext_aiche_papers_3298_p7.json
  Processing: ext_aiche_papers_3298_p8.json
  Processing: ext_aiche_papers_3299_p1.json
  Processing: ext_aiche_papers_3299_p2.json
  Processing: ext_aiche_papers_3299_p3.json
  Processing: ext_aiche_papers_3299_p4.json
  Processing: ext_aiche_papers_3299_p5.json
  Processing: ext_aiche_papers_3299_p6.json
  Processing: ext_aiche_papers_3299_p7.json
  Processing: ext_aiche_papers_3300_p1.json
  Processing: ext_aiche_papers_3300_p2.json
  Processing: ext_aiche_papers_3300_p3.json
  Processing: ext_aiche_papers_3302_p1.json
  Processing: ext_aiche_papers_3303_p1.json
  Processing: ext_aiche